# 01 - Pull TMDB

Downloads the TMDB film id export once, then pulls every chunk of films (5,000 ids each) in order. After each finished chunk the shard and the manifest go to the private Hub repo `Tsomaros/llm-movies-raw`, so nothing lives only on this Colab disk.
Use a **CPU runtime** (Runtime > Change runtime type > CPU). Needs the Colab Secrets `TMDB_READ_TOKEN` and `HF_TOKEN` (write access).

**To resume: rerun this notebook.** It downloads what the repo holds and carries on with the first chunk that is not done. A dead session loses only the chunk in progress.
The id export is downloaded once and kept in the repo, so the chunks stay the same for the whole pull. The last cell is **one chain** (`&&`): if any step fails, the rest do not run, so a failed download can never be followed by a pull that overwrites the repo's state. If it stops with an error, read it and run the cell again.

**A run with `LIMIT` or `MAX_CHUNKS` set is a sample.** `LIMIT` pulls only that many ids per chunk and leaves the chunk unmarked (a later full run finishes it); `MAX_CHUNKS` stops after that many chunks. Leave both empty for the full run.

## Setup

Run these cells top to bottom. They are the same in all three notebooks.

In [ ]:
# clone the repo the first time, update it after that
!test -d /content/LLM_Movies_Model/.git && git -C /content/LLM_Movies_Model pull || git clone https://github.com/Tsomaros/LLM_Movies_Model /content/LLM_Movies_Model
%cd /content/LLM_Movies_Model

In [ ]:
!pip install -q uv
!uv sync --frozen

## Secrets and environment

Reads `TMDB_READ_TOKEN` and `HF_TOKEN` (write access) from Colab Secrets, with notebook access switched on. They are never printed. `UV_FROZEN` stops later `uv run` calls from re-resolving the environment.

In [ ]:
import os

from google.colab import userdata

os.environ["TMDB_READ_TOKEN"] = userdata.get("TMDB_READ_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["UV_FROZEN"] = "1"

## Sample settings

Both empty means the full run.

In [ ]:
LIMIT = ""  #@param {type:"string"}
MAX_CHUNKS = ""  #@param {type:"string"}
LIMIT_FLAG = f"--limit {LIMIT}" if LIMIT else ""
MAX_CHUNKS_FLAG = f"--max-chunks {MAX_CHUNKS}" if MAX_CHUNKS else ""

## Pull every chunk that is not done

In order: download the id export from the repo, fetch it from TMDB only if the repo has none, upload it (refused if the repo already holds a different one), download the shards pulled so far, then pull.

In [ ]:
!uv run python -m llm_movies.data.hub_store download --dir /content/raw/ids --path-in-repo ids && uv run python -m llm_movies.data.tmdb_ids --out /content/raw/ids/movie_ids.json.gz && uv run python -m llm_movies.data.hub_store upload --dir /content/raw/ids --path-in-repo ids && uv run python -m llm_movies.data.hub_store download --dir /content/raw/tmdb --path-in-repo tmdb && uv run python -m llm_movies.data.tmdb_pull --ids /content/raw/ids/movie_ids.json.gz --all --repo Tsomaros/llm-movies-raw --out /content/raw/tmdb {LIMIT_FLAG} {MAX_CHUNKS_FLAG}